In [1]:
import os
import json
import time
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import evaluate
import onnxruntime as ort
from transformers import BlipProcessor, BlipForConditionalGeneration

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch: {torch.__version__}")
print(f"ONNX Runtime: {ort.__version__}")


c:\Users\nviquang\Documents\University\ThirdYear\Ky2\TTCS\Repo\SourceCode\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0605 22:12:34.844000 2508 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


PyTorch: 2.11.0+cu126
ONNX Runtime: 1.26.0


In [2]:
ROOT_DIR       = "../data_ready_for_kaggle"
TEST_PATH      = os.path.join(ROOT_DIR, "cleaned_test.csv")
IMAGES_DIR     = os.path.join(ROOT_DIR, "images_resized")
MODEL_DIR      = os.path.abspath("../saved_models_v3")
SAVE_DIR       = "./model_quantized/onnx_fp32"
RESULTS_DIR    = "./results"
SAMPLE_IMAGE   = "../data_ready_for_kaggle/images_resized/00000006091.jpg"

MAX_TEST_IMAGES = 200
MAX_LENGTH      = 64
BATCH_SIZE      = 16
NUM_WORKERS     = 0
WARMUP_STEPS    = 3

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Model dir : {MODEL_DIR}")
print(f"Save dir  : {SAVE_DIR}")


Model dir : c:\Users\nviquang\Documents\University\ThirdYear\Ky2\TTCS\Repo\SourceCode\saved_models_v3
Save dir  : ./model_quantized/onnx_fp32


In [3]:
processor = BlipProcessor.from_pretrained(MODEL_DIR)
model     = BlipForConditionalGeneration.from_pretrained(MODEL_DIR)
model.eval()

bos_token_id = model.config.text_config.bos_token_id
eos_token_id = model.config.text_config.eos_token_id
pad_token_id = model.config.text_config.pad_token_id
if pad_token_id is None:
    pad_token_id = eos_token_id

num_layers      = model.text_decoder.config.num_hidden_layers
num_heads       = model.text_decoder.config.num_attention_heads
head_dim        = model.text_decoder.config.hidden_size // num_heads
patch_size      = model.vision_model.config.patch_size
image_size      = model.vision_model.config.image_size
encoder_seq_len = (image_size // patch_size) ** 2 + 1

print(f"num_layers     : {num_layers}")
print(f"encoder_seq_len: {encoder_seq_len}")
print(f"bos/eos/pad    : {bos_token_id} / {eos_token_id} / {pad_token_id}")


Loading weights: 100%|██████████| 471/471 [00:00<00:00, 3258.29it/s]

num_layers     : 12
encoder_seq_len: 577
bos/eos/pad    : 0 / 2 / 1


In [4]:
class VisionEncoderWrapper(torch.nn.Module):
    """Chỉ forward vision encoder, trả về last_hidden_state."""
    def __init__(self, vision_model):
        super().__init__()
        self.vision_model = vision_model

    def forward(self, pixel_values):
        return self.vision_model(pixel_values=pixel_values, return_dict=True).last_hidden_state


class TextDecoderInitWrapper(torch.nn.Module):
    """Bước decode đầu tiên: không có past KV, trả về logits + KV cache."""
    def __init__(self, text_decoder, n_layers):
        super().__init__()
        self.text_decoder = text_decoder
        self.n_layers = n_layers

    def forward(self, input_ids, attention_mask, encoder_hidden_states, encoder_attention_mask):
        out = self.text_decoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            encoder_hidden_states=encoder_hidden_states,
            encoder_attention_mask=encoder_attention_mask,
            use_cache=True,
            return_dict=True,
        )
        tensors = [out.logits]
        self_layers  = out.past_key_values.self_attention_cache.layers
        cross_layers = out.past_key_values.cross_attention_cache.layers
        for i in range(self.n_layers):
            tensors += [
                self_layers[i].keys,  self_layers[i].values,
                cross_layers[i].keys, cross_layers[i].values,
            ]
        return tuple(tensors)


class TextDecoderWithPastWrapper(torch.nn.Module):
    """Các bước decode tiếp theo: nhận past KV, trả về logits + updated KV."""
    def __init__(self, text_decoder, n_layers):
        super().__init__()
        self.text_decoder = text_decoder
        self.n_layers = n_layers

    def forward(self, input_ids, attention_mask, encoder_hidden_states, encoder_attention_mask,
                *past_tensors):
        from transformers.cache_utils import DynamicCache, EncoderDecoderCache
        self_cache  = DynamicCache()
        cross_cache = DynamicCache()
        for i in range(self.n_layers):
            o = i * 4
            self_cache.update(past_tensors[o],     past_tensors[o + 1], i)
            cross_cache.update(past_tensors[o + 2], past_tensors[o + 3], i)
        cache = EncoderDecoderCache(self_cache, cross_cache)

        out = self.text_decoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            encoder_hidden_states=encoder_hidden_states,
            encoder_attention_mask=encoder_attention_mask,
            past_key_values=cache,
            use_cache=True,
            return_dict=True,
        )
        tensors = [out.logits]
        self_layers  = out.past_key_values.self_attention_cache.layers
        cross_layers = out.past_key_values.cross_attention_cache.layers
        for i in range(self.n_layers):
            tensors += [
                self_layers[i].keys,  self_layers[i].values,
                cross_layers[i].keys, cross_layers[i].values,
            ]
        return tuple(tensors)


vision_wrapper       = VisionEncoderWrapper(model.vision_model)
decoder_init_wrapper = TextDecoderInitWrapper(model.text_decoder, num_layers)
decoder_past_wrapper = TextDecoderWithPastWrapper(model.text_decoder, num_layers)
print("Wrappers created.")


Wrappers created.


In [5]:
ENCODER_PATH          = os.path.join(SAVE_DIR, 'vision_encoder_fp32.onnx')
DECODER_INIT_PATH     = os.path.join(SAVE_DIR, 'text_decoder_init_fp32.onnx')
DECODER_PAST_PATH     = os.path.join(SAVE_DIR, 'text_decoder_with_past_fp32.onnx')

image          = Image.open(SAMPLE_IMAGE).convert('RGB')
inputs         = processor(images=image, return_tensors='pt')
pixel_values   = inputs['pixel_values']
start_tokens   = torch.full((1, 1), bos_token_id, dtype=torch.long)
attn_mask      = torch.ones_like(start_tokens)

with torch.no_grad():
    enc_hidden = model.vision_model(pixel_values=pixel_values, return_dict=True).last_hidden_state
enc_attn_mask = torch.ones(enc_hidden.shape[:2], dtype=torch.long)

with torch.no_grad():
    init_out = decoder_init_wrapper(start_tokens, attn_mask, enc_hidden, enc_attn_mask)

past_in_names  = []
past_out_names = []
past_examples  = []
dyn_axes = {
    'pixel_values':          {0: 'batch'},
    'encoder_hidden_states': {0: 'batch', 1: 'enc_seq'},
    'encoder_attention_mask':{0: 'batch', 1: 'enc_seq'},
    'input_ids':             {0: 'batch', 1: 'dec_seq'},
    'attention_mask':        {0: 'batch', 1: 'dec_seq'},
    'logits':                {0: 'batch', 1: 'dec_seq'},
}
for i in range(num_layers):
    base = 1 + i * 4
    sk = f'past_self_key_{i}';   sv = f'past_self_val_{i}'
    ck = f'past_cross_key_{i}';  cv = f'past_cross_val_{i}'
    pk = f'pres_self_key_{i}';   pv = f'pres_self_val_{i}'
    qk = f'pres_cross_key_{i}';  qv = f'pres_cross_val_{i}'
    past_in_names  += [sk, sv, ck, cv]
    past_out_names += [pk, pv, qk, qv]
    past_examples  += list(init_out[base:base + 4])
    dyn_axes[sk] = {0:'batch', 2:'past_dec'}; dyn_axes[sv] = {0:'batch', 2:'past_dec'}
    dyn_axes[ck] = {0:'batch', 2:'enc_seq'};  dyn_axes[cv] = {0:'batch', 2:'enc_seq'}
    dyn_axes[pk] = {0:'batch', 2:'cur_dec'};  dyn_axes[pv] = {0:'batch', 2:'cur_dec'}
    dyn_axes[qk] = {0:'batch', 2:'enc_seq'};  dyn_axes[qv] = {0:'batch', 2:'enc_seq'}

print("Exporting vision encoder...")
torch.onnx.export(
    vision_wrapper, (pixel_values,),
    ENCODER_PATH,
    input_names=['pixel_values'],
    output_names=['encoder_hidden_states'],
    dynamic_axes={'pixel_values':{0:'batch'}, 'encoder_hidden_states':{0:'batch',1:'enc_seq'}},
    opset_version=17,
    do_constant_folding=True,
    dynamo=False,
)
print(f"  Saved: {ENCODER_PATH}")

print("Exporting text decoder init...")
torch.onnx.export(
    decoder_init_wrapper,
    (start_tokens, attn_mask, enc_hidden, enc_attn_mask),
    DECODER_INIT_PATH,
    input_names=['input_ids','attention_mask','encoder_hidden_states','encoder_attention_mask'],
    output_names=['logits'] + past_out_names,
    dynamic_axes=dyn_axes,
    opset_version=17,
    do_constant_folding=True,
    dynamo=False,
)
print(f"  Saved: {DECODER_INIT_PATH}")

print("Exporting text decoder with past...")
next_ids  = start_tokens[:, -1:]
next_mask = torch.ones((1, 2), dtype=torch.long)
torch.onnx.export(
    decoder_past_wrapper,
    (next_ids, next_mask, enc_hidden, enc_attn_mask, *past_examples),
    DECODER_PAST_PATH,
    input_names=['input_ids','attention_mask','encoder_hidden_states','encoder_attention_mask']
             + past_in_names,
    output_names=['logits'] + past_out_names,
    dynamic_axes=dyn_axes,
    opset_version=17,
    do_constant_folding=True,
    dynamo=False,
)
print(f"  Saved: {DECODER_PAST_PATH}")

processor.save_pretrained(SAVE_DIR)
print("\nAll 3 ONNX graphs exported successfully.")


Exporting vision encoder...


C:\Users\nviquang\AppData\Local\Temp\ipykernel_2508\293178081.py:44: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


  Saved: ./model_quantized/onnx_fp32\vision_encoder_fp32.onnx
Exporting text decoder init...


C:\Users\nviquang\AppData\Local\Temp\ipykernel_2508\293178081.py:57: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
c:\Users\nviquang\Documents\University\ThirdYear\Ky2\TTCS\Repo\SourceCode\.venv\Lib\site-packages\torch\onnx\_internal\torchscript_exporter\utils.py:1513: UserWarning: Provided key pixel_values for dynamic axes is not a valid input/output name
  _validate_dynamic_axes(dynamic_axes, model, input_names, output_names)
c:\Users\nviquang\Documents\University\ThirdYear\Ky2\TTCS\Repo\SourceCode\.venv\Lib\site-packages\torch\onnx\_internal\torchscript_exporter\utils.py:1513: UserWarning: Provided key past_self_key_0 for dynamic axes

  Saved: ./model_quantized/onnx_fp32\text_decoder_init_fp32.onnx
Exporting text decoder with past...


C:\Users\nviquang\AppData\Local\Temp\ipykernel_2508\293178081.py:73: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
c:\Users\nviquang\Documents\University\ThirdYear\Ky2\TTCS\Repo\SourceCode\.venv\Lib\site-packages\transformers\cache_utils.py:131: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if not self.is_initialized or self.keys.numel() == 0:
c:\Users\nviquang\Documents\University\ThirdYear\Ky2\TTCS\Repo\SourceCode\.ven

  Saved: ./model_quantized/onnx_fp32\text_decoder_with_past_fp32.onnx

All 3 ONNX graphs exported successfully.


In [6]:
for path in [ENCODER_PATH, DECODER_INIT_PATH, DECODER_PAST_PATH]:
    size_mb = os.path.getsize(path) / 1e6
    print(f"{os.path.basename(path):45s}  {size_mb:7.1f} MB")


vision_encoder_fp32.onnx                         344.5 MB
text_decoder_init_fp32.onnx                      851.5 MB
text_decoder_with_past_fp32.onnx                 794.8 MB


In [7]:
class UITViICDataset(Dataset):
    def __init__(self, data_file, img_dir, processor, max_length=64):
        df = pd.read_csv(data_file)
        self.img_dir   = img_dir
        self.processor = processor
        self.max_length = max_length
        self.image_groups = (
            df.groupby('image_file')['caption'].apply(list).reset_index()
        )

    def __len__(self):
        return len(self.image_groups)

    def __getitem__(self, idx):
        row        = self.image_groups.iloc[idx]
        image_file = row['image_file']
        image = Image.open(os.path.join(self.img_dir, image_file)).convert('RGB')
        enc = self.processor(
            images=image, return_tensors='pt',
            padding='max_length', truncation=True, max_length=self.max_length,
        )
        enc = {k: v.squeeze(0) for k, v in enc.items()}
        enc['image_file']   = image_file
        enc['raw_captions'] = row['caption']
        return enc


def collate_fn(batch):
    out = {k: torch.stack([b[k] for b in batch]) for k in ['pixel_values']}
    out['image_file']   = [b['image_file']   for b in batch]
    out['raw_captions'] = [b['raw_captions'] for b in batch]
    return out


def build_references(dataloader):
    refs = {}
    for batch in dataloader:
        for img, caps in zip(batch['image_file'], batch['raw_captions']):
            refs[img] = caps
    return refs


def calculate_metrics(preds_dict, refs_dict):
    common = sorted(set(preds_dict) & set(refs_dict))
    preds  = [preds_dict[k] for k in common]
    refs   = [refs_dict[k]  for k in common]
    bleu   = evaluate.load('bleu')
    rouge  = evaluate.load('rouge')
    meteor = evaluate.load('meteor')
    return {
        'bleu4':  bleu.compute(predictions=preds, references=refs)['bleu'] * 100,
        'rougeL': rouge.compute(predictions=preds, references=refs)['rougeL'] * 100,
        'meteor': meteor.compute(predictions=preds, references=refs)['meteor'] * 100,
        'num_images': len(common),
    }

print("Utilities defined.")


Utilities defined.


In [8]:
def select_provider():
    avail = ort.get_available_providers()
    if 'CUDAExecutionProvider' in avail:
        return ['CUDAExecutionProvider', 'CPUExecutionProvider'], 'cuda', 'CUDAExecutionProvider'
    return ['CPUExecutionProvider'], 'cpu', 'CPUExecutionProvider'


def greedy_decode_onnx(enc_sess, dec_init_sess, dec_past_sess,
                        pixel_values_np, max_length=64):
    """
    Greedy decode voi KV-cache.
    Inputs/Outputs kiem tra thuc te tu ONNX graph:
      decoder_init inputs : input_ids, attention_mask, encoder_hidden_states
                            (encoder_attention_mask bi constant-fold, khong truyen)
      decoder_past inputs : input_ids, attention_mask, past_self/cross_key/val_i
                            (encoder_hidden_states khong can, da bake vao cross-KV)
    QUAN TRONG: ham khong nhan processor lam tham so.
                processor la bien global trong notebook.
    """
    B = pixel_values_np.shape[0]

    enc_hidden = enc_sess.run(None, {'pixel_values': pixel_values_np})[0]

    generated = np.full((B, 1), bos_token_id, dtype=np.int64)
    finished  = np.zeros(B, dtype=bool)

    init_outs = dec_init_sess.run(None, {
        'input_ids':             generated,
        'attention_mask':        np.ones_like(generated, dtype=np.int64),
        'encoder_hidden_states': enc_hidden,
    })
    logits = init_outs[0]

    past = {}
    for i in range(num_layers):
        past[f'past_self_key_{i}']  = init_outs[1 + i*4]
        past[f'past_self_val_{i}']  = init_outs[2 + i*4]
        past[f'past_cross_key_{i}'] = init_outs[3 + i*4]
        past[f'past_cross_val_{i}'] = init_outs[4 + i*4]

    next_tok  = logits[:, -1, :].argmax(-1).astype(np.int64)
    next_tok  = np.where(finished, pad_token_id, next_tok)
    generated = np.concatenate([generated, next_tok[:, None]], axis=1)
    finished  = finished | (next_tok == eos_token_id)

    for _ in range(max_length - 2):
        if finished.all():
            break
        dec_in = {
            'input_ids':      next_tok[:, None],
            'attention_mask': np.ones((B, generated.shape[1]), dtype=np.int64),
        }
        dec_in.update(past)
        step_outs = dec_past_sess.run(None, dec_in)
        logits    = step_outs[0]
        for i in range(num_layers):
            past[f'past_self_key_{i}']  = step_outs[1 + i*4]
            past[f'past_self_val_{i}']  = step_outs[2 + i*4]
            past[f'past_cross_key_{i}'] = step_outs[3 + i*4]
            past[f'past_cross_val_{i}'] = step_outs[4 + i*4]
        next_tok  = logits[:, -1, :].argmax(-1).astype(np.int64)
        next_tok  = np.where(finished, pad_token_id, next_tok)
        generated = np.concatenate([generated, next_tok[:, None]], axis=1)
        finished  = finished | (next_tok == eos_token_id)

    return processor.batch_decode(generated, skip_special_tokens=True)


def run_benchmark(enc_sess, dec_init_sess, dec_past_sess,
                  dataloader, name, backend, precision, device, provider):
    preds_dict = {}
    latencies  = []
    for batch_idx, batch in enumerate(tqdm(dataloader, desc=f'Benchmarking {name}')):
        px = batch['pixel_values'].numpy().astype(np.float32)
        t0    = time.perf_counter()
        texts = greedy_decode_onnx(enc_sess, dec_init_sess, dec_past_sess, px, MAX_LENGTH)
        t1    = time.perf_counter()
        if batch_idx >= WARMUP_STEPS:
            latencies.append(t1 - t0)
        for img_f, text in zip(batch['image_file'], texts):
            preds_dict[img_f] = text.strip()
    return preds_dict, latencies


print('Benchmark functions defined.')


Benchmark functions defined.


In [9]:
test_dataset = UITViICDataset(TEST_PATH, IMAGES_DIR, processor, MAX_LENGTH)
test_dataset.image_groups = test_dataset.image_groups.head(MAX_TEST_IMAGES).reset_index(drop=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)

refs_dict = build_references(test_loader)
print(f"Test images: {len(test_dataset)}  |  batches: {len(test_loader)}")


Test images: 200  |  batches: 13


In [10]:
provider_list, bench_device, provider_name = select_provider()
sess_opts = ort.SessionOptions()
sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

enc_sess      = ort.InferenceSession(ENCODER_PATH,      sess_opts, providers=provider_list)
dec_init_sess = ort.InferenceSession(DECODER_INIT_PATH, sess_opts, providers=provider_list)
dec_past_sess = ort.InferenceSession(DECODER_PAST_PATH, sess_opts, providers=provider_list)

print(f'Provider: {provider_name}  |  Device: {bench_device}')

sample_px = processor(images=Image.open(SAMPLE_IMAGE).convert('RGB'),
                      return_tensors='pt')['pixel_values'].numpy().astype(np.float32)
_ = greedy_decode_onnx(enc_sess, dec_init_sess, dec_past_sess, sample_px)
print('Warmup done.')


Provider: CUDAExecutionProvider  |  Device: cuda
Warmup done.


In [11]:
preds_dict, latencies = run_benchmark(
    enc_sess, dec_init_sess, dec_past_sess,
    test_loader, 'onnx_fp32', 'onnx', 'fp32', bench_device, provider_name
)

metrics      = calculate_metrics(preds_dict, refs_dict)
avg_lat      = float(np.mean(latencies))    if latencies else 0.0
p95_lat      = float(np.percentile(latencies, 95)) if latencies else 0.0
throughput   = float(BATCH_SIZE / avg_lat)  if avg_lat > 0 else 0.0

benchmark = {
    'name':                           'onnx_fp32',
    'backend':                        'onnx',
    'precision':                      'fp32',
    'device':                         bench_device,
    'provider':                       provider_name,
    'generation_strategy':            'greedy+kvcache',
    'cache_enabled':                  True,
    'batch_size':                     BATCH_SIZE,
    'max_test_images':                MAX_TEST_IMAGES,
    'avg_latency_seconds_per_batch':  avg_lat,
    'p95_latency_seconds_per_batch':  p95_lat,
    'throughput_images_per_second':   throughput,
    'bleu4':                          metrics['bleu4'],
    'rougeL':                         metrics['rougeL'],
    'meteor':                         metrics['meteor'],
    'num_images':                     metrics['num_images'],
    'model_paths': {
        'vision_encoder':    ENCODER_PATH,
        'decoder_init':      DECODER_INIT_PATH,
        'decoder_with_past': DECODER_PAST_PATH,
    },
}

print(json.dumps(benchmark, indent=2, ensure_ascii=False))

with open(os.path.join(RESULTS_DIR, 'onnx_fp32.json'), 'w', encoding='utf-8') as f:
    json.dump(benchmark, f, ensure_ascii=False, indent=2)
print("\nSaved: results/onnx_fp32.json")


Benchmarking onnx_fp32: 100%|██████████| 13/13 [03:07<00:00, 14.45s/it]
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\nviquang\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\nviquang\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\nviquang\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


{
  "name": "onnx_fp32",
  "backend": "onnx",
  "precision": "fp32",
  "device": "cuda",
  "provider": "CUDAExecutionProvider",
  "generation_strategy": "greedy+kvcache",
  "cache_enabled": true,
  "batch_size": 16,
  "max_test_images": 200,
  "avg_latency_seconds_per_batch": 14.007376619999922,
  "p95_latency_seconds_per_batch": 17.965275350001136,
  "throughput_images_per_second": 1.1422552869146805,
  "bleu4": 22.826964076992915,
  "rougeL": 49.84811186344499,
  "meteor": 35.336904377949104,
  "num_images": 200,
  "model_paths": {
    "vision_encoder": "./model_quantized/onnx_fp32\\vision_encoder_fp32.onnx",
    "decoder_init": "./model_quantized/onnx_fp32\\text_decoder_init_fp32.onnx",
    "decoder_with_past": "./model_quantized/onnx_fp32\\text_decoder_with_past_fp32.onnx"
  }
}

Saved: results/onnx_fp32.json
